# **Air Quality Monitor Representativeness in the Contiguous United States**

#### **Objective:** Determine whether state and local governments site air monitors in areas that capture data that are representative of actual air quality.

#### **Scope:**

#### **Sources:**

| Num. | Title | Description | Source Link |
| :--- | :------------------------------ | :-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | :--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1. | _daily\_88101\_2019.csv_ | Derived from the U.S. EPA Pre\-Generated Data Files webpage on 10/28/2025, this dataset contains the PM2.5 air monitor data for every day in the year 2019 for all monitors in the United States and its territories;                                                                                                    \-\- Data Dictionary: [https://www.epa.gov/aqs/aqs\-data\-dictionary](https://www.epa.gov/aqs/aqs-data-dictionary) | [https://aqs.epa.gov/aqsweb/airdata/download\_files.html\#Daily](https://aqs.epa.gov/aqsweb/airdata/download_files.html#Daily) \(see "Particulates" and "2019"\) |
| 2. | USHAP\_PM2.5\_D1K\_2019\_V1.zip | Derived from research out of the Universities of Maryland and Iowa on 11/03/2025, this dataset contains 1\-kilometer resolution PM2.5 estimates in the contiguous United States for every day in the year 2019. The dataset was generated from big data \(e.g., ground\-based measurements, satellite remote sensing products, atmospheric reanalysis, and model simulations\) using artificial intelligence. The researchers show their estimates align well with physical measurements, with 0.82 cross\-validated coefficients of determination. | Paper: [https://www.thelancet.com/journals/lanplh/article/PIIS2542\-5196\(23\)00235\-8/fulltext](https://www.thelancet.com/journals/lanplh/article/PIIS2542-5196(23)00235-8/fulltext) <br> <br>  Data: <br> https://zenodo.org/records/7884640 |



#### Methodology

1. AQS Data Cleaning
   1. Download AQS Dataset
   2. Load Libraries
   3. Load AQS Dataset
   4. Select Relevant AQS Dataset Variables
   5. Generate a Dataset of PM2.5 Monitors with Locations
2. Alternative PM2.5 Values Dataset


#Add summary to check DF to make sure they make sense. 
#Consider lowercasing all variables in my dataset and in my variable names in R code
#Consider breaking scripts up into seperate jupyter notebooks
#Assess missingness


### **AQS Data Cleaning**



#### AQS Data Cleaning Pseudocode

1. Load libraries I will likely need
2. Load AQS dataset: DF &lt;\- read.csv
3. Limited\_DF &lt;\- Select\(State.Code, County.Code, Site.Num, Latitude, Longitude, Address, [State.Name](http://State.Name), [County.Name](http://County.Name), [City.Name](http://City.Name)\)
4. Unique identifier \(AQS\_ID\) &lt;\- Leading\_Zeros\_If\_Needed\(State.Code \+ County.Code  \+ Site.Num\)
5. Filter to unique AQS\_ID w/ locations



#### Load Libraries



In [1]:
#Purpose of Cell Block: Load libraries I will likely need
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(tidyr))
suppressPackageStartupMessages(library(readr))
suppressPackageStartupMessages(library(vroom))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(stringr))
suppressPackageStartupMessages(library(terra))

if (!require(dplyr)) install.packages("dplyr")
if (!require(dplyr)) install.packages("tidyr")
if (!require(readr)) install.packages("readr")
if (!require(vroom)) install.packages("vroom")
if (!require(ggplot2)) install.packages("ggplot2")
if (!require(stringr)) install.packages("stringr")
if (!require(stringr)) install.packages("terra")

library(dplyr)
library(tidyr)
library(readr)
library(vroom)
library(ggplot2)
library(stringr)
library(terra)

#### Alternative PM2.5 Data Cleaning Pseudocode

1. Change this



### **Monitors with Alternative PM2.5 Data \- Exploration of Dataset**



In [34]:
#Purpose of Cell Block: Load Processed AQM+AltPM25 Dataset 
setwd("/home/user/capstone/A_data")
DF_altpm25_processed <- read.csv("D_processed_data/alternative_PM25_data/alternative_PM25_data_processed_v01.csv") %>%
    mutate('AQS_Site_ID' = str_pad(AQS_Site_ID, width = 9, pad = 0)) %>% #pad leading zero so state code always 9 char
    mutate(Date = as.Date(Date)) #converting date value from character to date

head(DF_altpm25_processed)
DF_check <- DF_altpm25_processed %>%
    group_by(State) %>%
    summarise(Alt_PM25_Mean = mean(Alt_PM25))
DF_check



,AQS_Site_ID,Latitude,Longitude,Address,State,County,City,Date,Alt_PM25
,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<date>,<dbl>
1,011130003,32.43703,-84.99965,"510 6th Place South, Phenix City, Alabama 36869",Alabama,Russell,Phenix City,2019-01-01,4.8
2,020200018,61.20586,-149.82460,3000 EAST 16TH AVENUE,Alaska,Anchorage,Anchorage,2019-01-01,NA
3,020201004,61.32670,-149.56971,11723 OLD GLENN HIGHWAY,Alaska,Anchorage,Anchorage,2019-01-01,NA
4,020900010,64.84067,-147.72246,"675 7TH AVENUE, FAIRBANKS",Alaska,Fairbanks North Star,Fairbanks,2019-01-01,NA
5,020900034,64.84569,-147.72741,907 Terminal St.,Alaska,Fairbanks North Star,Fairbanks,2019-01-01,NA
6,020900035,64.76264,-147.31028,"3288 Hurst Road, North Pole, Alaska 99705",Alaska,Fairbanks North Star,Not in a city,2019-01-01,NA


State,Alt_PM25_Mean
<chr>,<dbl>
Alabama,8.402488
Alaska,NA
Arizona,6.526979
Arkansas,8.701195
California,NA
Colorado,6.802522
Connecticut,6.990038
Delaware,7.423113
District Of Columbia,8.311379


In [25]:
#Purpose of Cell: Figureout what went wrong with NC, TX, WA, and WI

setwd("/home/user/capstone/A_data")
altPM25_files <- list.files("C_raw_data/alternative_PM25_data", pattern = "\\.nc$", full.names = TRUE) #identify the number of alt. PM2.5 files in data folder
DF_aqs2019_processed <- read.csv("D_processed_data/air_monitors/daily_88101_2019_processed.csv")

#Creating a function that converts the dates in the filenames into a string, which we can use to add our alternative PM2.5 variables to our daily monitor data
AltPM25_filename_date <- function(file_path) {
    file_name <- basename(file_path) #extract filename
    sDate <- regmatches(file_name, regexpr("[0-9]{8}", file_name)) #extract filenames' date
    as.Date(sDate, format = "%Y%m%d") #convert to date (storing it in sDate, which is a string)
}

#For each day of alternative data, grab the alternative date and use it to add alternative data to the associated day for each monitor
for (AltPM25_file in altPM25_files) { 
    AltDate <- AltPM25_filename_date(AltPM25_file)
    
    #Smaller helper to keep me updated on the for loop's progress (typical runtime is several minutes)
#     i <- i + 1
#     cat("\rProcessing", i, "of", length(altPM25_files),
#         "| Date:", format(AltDate), "       ")
#     flush.console()
    
    #Filter to only monitors on the date associated with our alternative PM2.5 dataset
    DF_monitors_on_date_subset <- DF_aqs2019_processed %>%
        filter(Date == AltDate) %>%
        filter(State %in% c("North Carolina", "Texas", "Washington", "Wisconsin"))
    
    #Prepare plot our filtered monitors 
    SPAC_points_on_date <- vect(DF_monitors_on_date_subset %>% 
        select(AQS_Site_ID, Longitude, Latitude),
        geom = c("Longitude", "Latitude"),
        crs  = "EPSG:4326"
    )

    #Check total number of monitors
    DF_points_on_date <- as.data.frame(SPAC_points_on_date)
    
    total_points_on_date <- DF_points_on_date %>%
        summarise(total_sites = n_distinct(AQS_Site_ID))
    
    #Prepare plot for our alternative air quality data
    SPAC_alt_air_data_on_date <- rast(AltPM25_file)

    #Extract alternative PM2.5 value at monitor locations 
    DF_altPM25 <- extract(SPAC_alt_air_data_on_date, 
                          SPAC_points_on_date, 
                          ID = FALSE)
    
    varname <- names(DF_altPM25)[1] #Extract variable name from alternative dataset
    DF_monitors_on_date_subset$Alt_PM25 <- DF_altPM25[[varname]]
    
}

#sanity check that extraction worked
print(head(DF_monitors_on_date_subset))

#Preserve output in processed data folder
write_csv(DF_monitors_on_date_subset, "D_processed_data/alternative_PM25_data/alternative_PM25_data_subset_processed.csv")
print("Successfully exported dataset!")

  AQS_Site_ID Latitude Longitude                 Address          State
1   370210034 35.60620 -82.58440        175 BINGHAM ROAD North Carolina
2   370350004 35.72889 -81.36556         1650 1ST STREET North Carolina
3   370510009 35.04142 -78.95311         4533 RAEFORD RD North Carolina
4   370570002 35.81450 -80.26270      938 S.SALISBURY ST North Carolina
5   370630015 36.03296 -78.90404       801 STADIUM DRIVE North Carolina
6   370670022 36.11094 -80.22450 1300 BLK. HATTIE AVENUE North Carolina
      County          City       Date Alt_PM25
1   Buncombe     Asheville 2019-12-31      1.6
2    Catawba       Hickory 2019-12-31      0.6
3 Cumberland  Fayetteville 2019-12-31      3.3
4   Davidson     Lexington 2019-12-31      3.5
5     Durham        Durham 2019-12-31      3.4
6    Forsyth Winston-Salem 2019-12-31      2.7


[1] "Successfully exported dataset!"


In [27]:
#Purpose of Cell Block: Load Processed AQM+AltPM25 Dataset 
head(DF_monitors_on_date_subset)
DF_check_subset <- DF_monitors_on_date_subset %>%
    group_by(State) %>%
    summarise(Alt_PM25_Mean = mean(Alt_PM25))
DF_check_subset

,AQS_Site_ID,Latitude,Longitude,Address,State,County,City,Date,Alt_PM25
,<int>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
1,370210034,35.60620,-82.58440,175 BINGHAM ROAD,North Carolina,Buncombe,Asheville,2019-12-31,1.6
2,370350004,35.72889,-81.36556,1650 1ST STREET,North Carolina,Catawba,Hickory,2019-12-31,0.6
3,370510009,35.04142,-78.95311,4533 RAEFORD RD,North Carolina,Cumberland,Fayetteville,2019-12-31,3.3
4,370570002,35.81450,-80.26270,938 S.SALISBURY ST,North Carolina,Davidson,Lexington,2019-12-31,3.5
5,370630015,36.03296,-78.90404,801 STADIUM DRIVE,North Carolina,Durham,Durham,2019-12-31,3.4
6,370670022,36.11094,-80.22450,1300 BLK. HATTIE AVENUE,North Carolina,Forsyth,Winston-Salem,2019-12-31,2.7


State,Alt_PM25_Mean
<chr>,<dbl>
North Carolina,NA
Texas,NA
Washington,NA
Wisconsin,NA
